In [3]:
import os
import torch
import warnings
import chunking as C
from tqdm import tqdm
from pathlib import Path
from torch import nn, optim
from resnet20 import resnet20
import torch.nn.functional as F
import json, random, numpy as np
from torchvision import transforms
from torch.utils.data import DataLoader
from torchvision.datasets import CIFAR100
from torch.utils.data import DataLoader, Dataset
from constants import SEED, device, TOKENS_PER_SEQ
warnings.filterwarnings("ignore")

### Extract every expert checkpoint, the backbone is handled separately

In [23]:

rows = []
models_path = Path('res_models')

EXCLUDE_NAMES = {'backbone_a.pt'}

ckpt_paths = sorted(
    p for p in models_path.rglob('*.pt')
    if 'resume' not in p.name and p.name not in EXCLUDE_NAMES
)

for path in tqdm(ckpt_paths, desc='extracting'):
    ck = torch.load(path, map_location='cpu')

    '''strict load so a checkpoint that does not match the architecture fails here'''
    model = resnet20(num_classes=100)
    model.load_state_dict(ck['state_dict'], strict=True)
    sd = model.state_dict()

    chunks, mask, seq_index, meta = C.extract_model(
        sd, ck['model_id'], checkpoint_path=str(path), split_id=ck['split_id'],
        seed=ck['seed'], epoch=ck['epoch'], init_group=ck['init_group']
    )

    C.verify_alignment(meta)
    C.verify_coverage(meta, sd)
    C.verify_roundtrip(chunks, meta, sd, verbose=False)

    C.save(f'./zoo_chunks/{ck["model_id"]}', chunks, mask, seq_index, meta)
    rows.append(ck['model_id'])

assert len(rows) == len(set(rows)), 'duplicate model_id, checkpoints would overwrite each other'
print(f'saved {len(rows)} expert models to ./zoo_chunks')

extracting: 100%|██████████| 50/50 [01:48<00:00,  2.17s/it]

saved 50 expert models to ./zoo_chunks


### Backbone extracted separately, it is a reference model not zoo training data

In [ ]:
ck = torch.load('res_models/backbone_a.pt', map_location='cpu')
model = resnet20(num_classes=100)
model.load_state_dict(ck['state_dict'], strict=True)
sd = model.state_dict()

chunks, mask, seq_index, meta = C.extract_model(
    sd, 'backbone_a', checkpoint_path='res_models/backbone_a.pt',
    split_id='full', seed=ck['seed'], epoch=ck['epoch'], init_group='base_a'
)
C.verify_alignment(meta); C.verify_coverage(meta, sd)
C.verify_roundtrip(chunks, meta, sd, verbose=False)
C.save('./zoo_reference/backbone_a', chunks, mask, seq_index, meta)
print('backbone saved to ./zoo_reference/backbone_a')

backbone saved to ./zoo_reference/backbone_a


In [4]:
class ResZoo(Dataset):
    def __init__(self, root_dir='./zoo_chunks', model_ids=None):
        self.root_dir = root_dir
        '''filter to an explicit split, never load the whole directory blindly'''
        if model_ids is None:
            self.splits = sorted(os.listdir(root_dir))
        else:
            self.splits = sorted(model_ids)

        self.ids, self.chunks_list, self.mask_list, self.seq_index_list, self.meta_list = [], [], [], [], []
        '''flat index of (model_idx, row_start) so one item is one sequence'''
        self.index = []

        for split in self.splits:
            path = os.path.join(root_dir, split)
            chunks, mask, seq_index, meta = self.load_split_chunk(path)
            m = len(self.ids)
            self.ids.append(split)
            self.chunks_list.append(chunks)
            self.mask_list.append(mask)
            self.seq_index_list.append(seq_index)
            self.meta_list.append(meta)

            '''one entry per sequence, tokens_per_seq rows each'''
            T = meta.tokens_per_seq
            for start in range(0, chunks.shape[0], T):
                self.index.append((m, start))

        self.tokens_per_seq = self.meta_list[0].tokens_per_seq
        self.chunk_size = self.meta_list[0].chunk_size

        '''map each row back to its layer, needed for conditioning embeddings'''
        self.row_layer = []
        for meta in self.meta_list:
            lut = np.zeros(meta.n_chunks_total, dtype=np.int64)
            for li, lm in enumerate(meta.layers):
                lut[lm.chunk_start:lm.chunk_end] = li
            self.row_layer.append(lut)

    def load_split_chunk(self, path):
        chunks, mask, seq_index, meta = C.load(path)
        return chunks, mask, seq_index, meta

    def __len__(self):
        '''one item is one sequence, not one model'''
        return len(self.index)

    def __getitem__(self, idx):
        m, start = self.index[idx]
        end = start + self.tokens_per_seq
        lm_ids = self.row_layer[m][start:end]
        meta = self.meta_list[m]
        '''depth and stage of the layer this sequence came from'''
        depth = meta.layers[int(lm_ids[0])].depth_index
        stage = meta.layers[int(lm_ids[0])].stage
        return {
            'chunks': torch.from_numpy(self.chunks_list[m][start:end]).float(),
            'mask': torch.from_numpy(self.mask_list[m][start:end]).bool(),
            'depth': torch.tensor(depth, dtype=torch.long),
            'stage': torch.tensor(stage, dtype=torch.long),
            'model_idx': torch.tensor(m, dtype=torch.long),
            'row_start': torch.tensor(start, dtype=torch.long),
        }